In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# 方針
これまで作ってきた特徴量を追加し、Pipelineの中に組み込んでいく。そして、いくつかのモデルのスコアをパラメータの最適解を探りながら算出し、それらモデルのスコア比較を行い、最適モデルを見つける

* 「03_eda」,「08_feature_engineering」ファイルより、新特徴量として、”Sex_Pclass"、"logFare"、"len_Fam"、"Title"、”surb_rank"、"has_cabin"、"cabin_deck"の7つを加える
* 「07_pipeline」の中に組み込んでいく

In [2]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


# データの読み込み

In [3]:
train_csv = pd.read_csv("/kaggle/input/competitions/titanic/train.csv").set_index("PassengerId")
test_csv = pd.read_csv("/kaggle/input/competitions/titanic/test.csv").set_index("PassengerId")

y = train_csv.Survived
X = train_csv.drop("Survived", axis=1)

# カスタム特徴量を作るトランスフォーマー
「08_feature_engineering」ファイルでつくりだした特徴量である、"Title","surv_rank","has_cabin","cabin_deck"の4つを追加していく

※「len_fam」について
>前回まではlen_famの値として,家族の合計によって"alone"、"basic"、"large"の3つに分けていたが、これらの値は本来であれば大小関係を数値によって学習できるものになるので、カテゴリとして処理するよりも数値列として学習させたほうが良いと判断。よって、今回は｛"alone":1,"basic":2,"large":3｝として、それぞれ重みを付けた形で再定義した。

In [4]:
class TitanicFeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass #paramatorはない

    def fit(self,X,y=None):
        #ここでいうXは、渡されたX、つまりcross_val時に渡されたtrain_X
        #統計量(medianなど)を使う場合はここで定義しないとデータリークにつながる
        X = X.copy()
        
        X["Title"] = X["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
        X["Title"] = X["Title"].replace(["Mlle", "Ms"], "Miss")
        X["Title"] = X["Title"].replace("Mme", "Mrs")
        rare_titles = [
            "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
            "Rev", "Sir", "Jonkheer", "Dona"
        ]
        X["Title"] = X["Title"].replace(rare_titles, "Rare")

        self.title_age_median = X.groupby("Title")["Age"].median()
        self.global_age_median = X["Age"].median()

        return self
        
    def transform(self,X): #ここから新特徴量の作成
        X_new = X.copy() 
        #ここのXはデータ全体である"X"なので、これを使った統計量を使うと
        #データリークをしてしまう点に注意

        #"Sex_Pclass"の作成 
        X_new["Sex_Pclass"] = (
            X_new["Sex"].astype(str) + "_" +
            X_new["Pclass"].astype(str)
        )

        #"logFare"の作成
        X_new["logFare"] = np.log1p(X_new["Fare"])

        #"len_fam"の作成
        family = X_new["Parch"] + X_new["SibSp"]
        X_new["len_fam"] = family.apply(
            lambda x: (
                0 if x == 0 else
                1 if 1<=x<=3 else
                2 
            )
        )

        #"Title"の作成
        X_new["Title"] = X_new["Name"].str.extract(r" ([A-Za-z]+)\.",expand = False)
        X_new["Title"] = X_new["Title"].replace(["Mlle","Ms"],"Miss")
        X_new["Title"] = X_new["Title"].replace("Mme","Mrs")
        rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
        X_new["Title"] = X_new["Title"].replace(rare_titles,"Rare")

        #”Age"の再定義。年齢の欠損を敬称別の中央値の年齢で埋める
        #train用のデータで学習した中央値で代入。また、validに未知のTitleがある場合は、全体平均を使用
        X_new["Age"] = X_new["Age"].fillna(X_new["Title"].map(self.title_age_median))
        X_new["Age"] = X_new["Age"].fillna(self.global_age_median)

        #データリークのため、この特徴量を削除
        #追記：データリークではなかったが、年齢による線引きを変更
        #"surv_rank"を作成、デフォルトで0点上記の条件以外の部分
        condition_2points = ( #子供最優先
            ((X_new["Sex"]=="male") & (X_new["Age"]<12)) |
            ((X_new["Sex"]=="female") & ((X_new["Age"]<12) ))
        )
        condition_1point = ( #次に全女性優先
            (X_new["Sex"]=="female") & (X_new["Age"]>=12)
        )
        
        conditions = [condition_2points,condition_1point]
        points = [2,1]
        X_new["surv_rank"] = np.select(conditions,points,default=0)

        #"has_cabin"を作成
        X_new["has_cabin"] = X_new["Cabin"].notnull().astype(int)

        #"cabin_deck"を作成
        X_new["cabin_deck"] = X_new["Cabin"].str[0].fillna("Unknown")
        X_new.loc[X_new["cabin_deck"].isin(["G","T"]),"cabin_deck"] = "Unknown"
        
        return X_new
        

# Pipelineの構築
今回は複数モデルの比較を行いたいため、modelをパイプラインから除外し、feature_engineering,preprocessorのみをパイプラインの中に残す

In [5]:
#数値列に対する前処理
numerical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

#カテゴリ列に対する前処理
categorical_transformer = Pipeline(
    steps = [
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output = False))
    ]
)

#使用する列を明示、"len_fam"は今回はnum_colsに移動している
num_cols = ["logFare","len_fam","has_cabin","surv_rank"]
cat_cols = ["Embarked","Sex_Pclass","Title"]

#前処理の結合
preprocessor = ColumnTransformer(
    transformers=[
        ("num",numerical_transformer,num_cols),
        ("cat",categorical_transformer,cat_cols)
    ]
)


# 各モデル+OPtunaの使用
各モデルの予測と、Optunaを用いたパラメータの最適解を算出していく。また。cros_val_scoreの分割にランダム性をもせて、Optunaの学習時に過学習を防ぎ、汎用性を持たせるようにする。

In [6]:
#cros_val_score時の分割時の詳細設定
from sklearn.model_selection import StratifiedKFold

cv =StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ①LogisticRegressionの場合

In [7]:
from sklearn.linear_model import LogisticRegression

Log_pipeline = Pipeline(
    steps = [
        ("feature_engineering",TitanicFeatureEngineering()),
        ("preprocessor",preprocessor),
        ("model",LogisticRegression(max_iter=1000))
    ]
)


ここから、Optunaを用いた、パラメータの最適解を探る。

◆LogisticRegressionのパラメータについて

* 「C」：「正則化の強さの逆数」C が大きいほど正則化が弱くなり、C が小さいほど正則化が強くなる。つまり、Cが大きいと過学習しやすくなり、Cが小さいと汎化性能が上がる。
  >0.001 ～ 10
特に 0.01 / 0.1 / 1 / 10 がよく使われる

In [8]:
import optuna
from sklearn.model_selection import cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING) #途中式を消すため

#評価関数(目的関数)の定義
def LR_func(trial):
    log_c = trial.suggest_float("model__C",1e-4,1e4,log=True) #正則化パラメータ「c」の探索範囲を設定、logで対数スケーリング

    Log_pipeline.set_params(model__C=log_c) #モデル内の「c」の要素にlog_c書き換え

    scores = cross_val_score( #クロス検証
        Log_pipeline,X,y,cv=cv,scoring="accuracy",n_jobs=-1
    )

    return np.mean(scores) #5回分の交差検証のスコア平均点を返す

study = optuna.create_study(direction="maximize") #Accuracyを最大化する設定

study.optimize(LR_func,n_trials=30) #パラメータ探索

print("最適なパラメータ：", study.best_params)
print("最適解のスコア：", study.best_value)

LogisticRegression_best_value = study.best_value


最適なパラメータ： {'model__C': 3770.78026775709}
最適解のスコア： 0.8327349193396522


# ②RandomForestClassifierの場合

In [9]:
from sklearn.ensemble import RandomForestClassifier

RFC_pipeline = Pipeline(
    steps = [
        ("feature_engineering",TitanicFeatureEngineering()),
        ("preprocessor",preprocessor),
        ("model",RandomForestClassifier(
            n_jobs=-1,
            #random_state=10,
        ))
    ]
)


「Optuna」を使ったパラメータの最適解を探っていく

◆LogisticRegressionのパラメータについて

* n_estimators:決定木の本数。多いほど安定するが、計算コストが増える
  >典型的な範囲:
小さめのタスク: 100〜300
余裕があるなら: 300〜500 程度
* max_depth:各決定木の最大の深さ。深くすると複雑になり過学習しやすい／浅くすると単純になりすぎる
  >典型的な範囲: 5〜20
* min_samples_leaf:葉ノードに必要な最小サンプル数。値を大きくすると「1サンプルだけの葉」を避け、過学習を抑える
  >典型的な範囲: 1〜10
* min_samples_split:内部ノードを分割するのに必要な最小サンプル数。値を大きくすると、サンプルが少ないノードを細かく分割しなくなり、木が単純になる
  >典型的な範囲: 2〜10

In [10]:
def RFC_func(trial):
    rf_n_estimators = trial.suggest_int("rf_n_estimators", 100, 1000, step=100)
    rf_max_depth = trial.suggest_int("rf_max_depth", 3, 15)
    rf_min_samples_leaf = trial.suggest_int("rf_min_samples_leaf",2,10)
    rf_min_samples_split = trial.suggest_int("rf_min_samples_split", 2, 10)

    RFC_pipeline.set_params(
        model__n_estimators = rf_n_estimators,
        model__max_depth = rf_max_depth,
        model__min_samples_leaf = rf_min_samples_leaf,
        model__min_samples_split = rf_min_samples_split
    )

    scores = cross_val_score(
        RFC_pipeline,X,y,cv=cv,scoring="accuracy",n_jobs=-1
    )

    return np.mean(scores)


study = optuna.create_study(direction = "maximize")
study.optimize(RFC_func,n_trials = 30)

print("最も高精度だったパラメータ:", study.best_params)
print("最適解のスコア (Accuracy):", study.best_value)

最も高精度だったパラメータ: {'rf_n_estimators': 200, 'rf_max_depth': 15, 'rf_min_samples_leaf': 5, 'rf_min_samples_split': 6}
最適解のスコア (Accuracy): 0.8372481325717155


In [11]:
RFC_best_value = study.best_value
print(RFC_best_value)

#得た最適なパラメータとX,yを使ってモデルを学習させる
RFC_pipeline.set_params(
    model__n_estimators = study.best_params["rf_n_estimators"],
    model__max_depth = study.best_params["rf_max_depth"],
    model__min_samples_leaf = study.best_params["rf_min_samples_leaf"],
    model__min_samples_split = study.best_params["rf_min_samples_split"]
)

RFC_pipeline.fit(X,y)

0.8372481325717155


Pipeline(steps=[('feature_engineering', TitanicFeatureEngineering()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['logFare', 'len_fam',
                                                   'has_cabin', 'surv_rank']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked', 'Sex_Pclass',
                                                   'Title'])])),
                ('model',
                 RandomForestClassifier(max_depth=15, min_samples_leaf=5,
                                        min_samples_split=6, n_estimators=200,
                                        n_jobs=-1))])

# ③XGBClassifierの場合

In [12]:
from xgboost import XGBClassifier

XGBC_pipeline = Pipeline(steps = [
        ("feature_engineering",TitanicFeatureEngineering()),
        ("preprocessor",preprocessor),
        ("model",XGBClassifier(
            n_jobs=-1,
            #random_state=10,
            eval_metric = "logloss" #警告文を消すため
        ))
    ]
)


Optunaで最適なパラメータを探索

◆XGBClassifierのパラメータについて(RFCと同じものは説明除外)

* learning_rate:1 本の木がどれだけ強く予測に影響するか（木の寄与度）。
  >実務的な範囲
0.01 ～ 0.3　デフォルトは 0.3（やや大きい）
* subsample:各木を作るときに使うデータの割合（行方向のサンプリング）
  >実務的な範囲
0.6 ～ 1.0


In [13]:
def XGBC_func(trial):
    xgb_learning_rate = trial.suggest_float("xgb_learning_rate",0.01,0.3,log=True)
    xgb_max_depth = trial.suggest_int("xgb_max_depth",3,10)
    xgb_n_estimators = trial.suggest_int("xgb_n_estimators",100,1000,step=100)
    xgb_subsample = trial.suggest_float("xgb_subsample",0.6,1.0)

    XGBC_pipeline.set_params(
        model__learning_rate = xgb_learning_rate,
        model__max_depth = xgb_max_depth,
        model__n_estimators = xgb_n_estimators,
        model__subsample = xgb_subsample
    )

    scores = cross_val_score(XGBC_pipeline,X,y,cv=cv,scoring="accuracy",n_jobs=-1)
    return np.mean(scores)

study = optuna.create_study(direction = "maximize")
study.optimize(XGBC_func, n_trials=30)

print("最も高精度だったパラメータ:", study.best_params)
print("最適解のスコア (Accuracy):", study.best_value)

XGBC_best_value=study.best_value

最も高精度だったパラメータ: {'xgb_learning_rate': 0.0103618060656067, 'xgb_max_depth': 6, 'xgb_n_estimators': 300, 'xgb_subsample': 0.6123091939857642}
最適解のスコア (Accuracy): 0.8451007469713139


# 各モデルの性能比較

In [14]:
pd.Series({"LogisticRegression_best_value":LogisticRegression_best_value,
             "RFC_best_value":RFC_best_value,
             "XGBC_best_value":XGBC_best_value},name="モデルのスコア比較").sort_values(ascending=False)

XGBC_best_value                  0.845101
RFC_best_value                   0.837248
LogisticRegression_best_value    0.832735
Name: モデルのスコア比較, dtype: float64

このことから、この3つのモデルにおいて、今回行ったfeature_engineering、前処理などの条件下においての最適なモデルは"XGBClassifier"であることがわかった。

XGBClassifierのモデルを最適パラメータで学習させたのちに、testデータで予測を作成していく

In [15]:
XGBC_pipeline.set_params(
    model__learning_rate=study.best_params["xgb_learning_rate"],
    model__max_depth=study.best_params["xgb_max_depth"],
    model__n_estimators=study.best_params["xgb_n_estimators"],
    model__subsample=study.best_params["xgb_subsample"]
)

XGBC_pipeline.fit(X,y)


preds = XGBC_pipeline.predict(test_csv)

output = pd.DataFrame({"PassengerId":test_csv.index,"Survived":preds})
output.to_csv("submission.csv",index=False)

# スコア改善ログ
* 1回目の提出0.772：なぜか、以前のパラメータ調整を行っていないscoreと全く同じ0.772になった。全く同じなのはおかしいので理由を探っていく。とりあえずpipelineの中のrandom_stateを入れてしまっているせいで、乱数固定になっているのかと思い、random_stateをアンコメントした。
* 2回目の提出0.763：スコアがパラメータ調整をしていないときより低くなって0.763となった。これは、データリークの可能性があると判定。作った特徴量を見直すと、"surv_rank"がそもそもデータリークになっている可能性。
* "surv_rank"事態は、データリークに該当しなかった。おそらく、6歳、12歳、48歳という年齢を詳細に線引き(境界線決め)してしまったために、過学習してしまった結果が、CVとLBの点数の差になっている。
  >解決策:"surv_rank"の年齢による線引きをもっと大雑把に変更
* "has_cabin"がカテゴリ列に分類してしまっていた
  >数値列に配置しなおし。
* cross_val時に、"Age"列の欠損を中央値で埋めるときに、データリークの可能性。
>元コード：X_new["Age"] = X_new["Age"].fillna(X_new.groupby("Title")["Age"].transform("median"))
> .transform()を使うとcross_val時に検証用データも見た中央値を使ってしまっている。
>同様に、"Title"列も、valid側の情報を見てしまう可能性があるので、class内のdef fit(self, X, y=None):にX_trainの情報だけ抜き出してデータリークを回避する。
* cross_val_scoreの分割に偏りがある可能性
  >shuffle = True でデータの並びをバラバラに。PassengerIdをインデックスにしているので、shuffleしていないと間接的に並び順が特徴量として影響を与えていたかも。
* cros_val_scoreだけ0.845を超えて以上に高いので、やはり特徴量に問題があるとみなす。過学習の温床の可能性のあるものを削除
  >surv_rankの削除


In [16]:
import pandas as pd

X_pre = XGBC_pipeline.named_steps["feature_engineering"].fit_transform(X)

X_encoded = XGBC_pipeline.named_steps[
    "preprocessor"
].fit_transform(X_pre)

feature_names = XGBC_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

X_final = pd.DataFrame(
    X_encoded,
    columns=feature_names
)

X_final.columns

Index(['num__logFare', 'num__len_fam', 'num__has_cabin', 'num__surv_rank',
       'cat__Embarked_C', 'cat__Embarked_Q', 'cat__Embarked_S',
       'cat__Sex_Pclass_female_1', 'cat__Sex_Pclass_female_2',
       'cat__Sex_Pclass_female_3', 'cat__Sex_Pclass_male_1',
       'cat__Sex_Pclass_male_2', 'cat__Sex_Pclass_male_3', 'cat__Title_Master',
       'cat__Title_Miss', 'cat__Title_Mr', 'cat__Title_Mrs',
       'cat__Title_Rare'],
      dtype='object')